In [2]:
#checking longitude and latitude extent in negative dataset to be sure they fall within Canada's borders

import pandas as pd
df_unlabeled = pd.read_csv("data_combined/unlabeled_samples.csv")

# check spatial extent
print("Longitude range:", df_unlabeled["longitude"].min(), "to", df_unlabeled["longitude"].max())
print("Latitude range:", df_unlabeled["latitude"].min(), "to", df_unlabeled["latitude"].max())
print("Shape:", df_unlabeled.shape)

Longitude range: -150.3687 to -40.353214
Latitude range: 43.402084 to 74.69019
Shape: (128380, 33)


In [3]:
# Canada's approximate bounding box
LON_MIN = -141.0
LON_MAX = -52.0
LAT_MIN = 41.7
LAT_MAX = 83.0



df_unlabeled_clipped = df_unlabeled[
    (df_unlabeled["longitude"] >= LON_MIN) &
    (df_unlabeled["longitude"] <= LON_MAX) &
    (df_unlabeled["latitude"] >= LAT_MIN) &
    (df_unlabeled["latitude"] <= LAT_MAX)
]

print("Before:", len(df_unlabeled))
print("After:", len(df_unlabeled_clipped))

Before: 128380
After: 123710


In [4]:
df_mines = pd.read_csv("data_cleaned/mines.csv")

# clip unlabeled to canada bounding box
df_unlabeled = df_unlabeled[
    (df_unlabeled["longitude"] >= -141.0) &
    (df_unlabeled["longitude"] <= -52.0) &
    (df_unlabeled["latitude"] >= 41.7) &
    (df_unlabeled["latitude"] <= 83.0)
].copy()

# target minerals
target_minerals = [
    'Cobalt', 'Copper', 'Diamond', 'Gold', 'Graphite',
    'Lithium', 'Nickel', 'Rare Earth Elements (REE)', 'Silver', 'Uranium'
]

# feature columns
feature_cols = [
    'Vp_100_kms', 'MagSus_SI', 'Density_gcc',
    'SiO2_pct', 'Al2O3_pct', 'Fe2O3_pct', 'MgO_pct', 'CaO_pct',
    'Na2O_pct', 'K2O_pct', 'TiO2_pct', 'P2O5_pct', 'MnO_pct',
    'Au_ppb', 'Cu_ppm', 'Ni_ppm', 'Co_ppm', 'Li_ppm', 'Cr_ppm',
    'Zn_ppm', 'Pb_ppm', 'Mo_ppm', 'As_ppm', 'Th_ppm', 'U_ppm',
    'La_ppm', 'Ce_ppm', 'Nd_ppm', 'Nb_ppm', 'V_ppm', 'W_ppm'
]

# build positive samples with per-mineral labels
positives = df_mines[df_mines[target_minerals].any(axis=1)].copy()
positives = positives[["longitude", "latitude"] + feature_cols + target_minerals].copy()

# unlabeled samples get 0 for all mineral labels
unlabeled = df_unlabeled[["longitude", "latitude"] + feature_cols].copy()
for mineral in target_minerals:
    unlabeled[mineral] = 0

# combine
df_combined = pd.concat([positives, unlabeled], ignore_index=True)

df_combined.to_csv("data_cleaned/pul_dataset.csv", index=False)

In [5]:
print(df_combined[target_minerals].sum())

Cobalt                        44
Copper                       276
Diamond                        9
Gold                         425
Graphite                       4
Lithium                        4
Nickel                        59
Rare Earth Elements (REE)      2
Silver                       281
Uranium                       22
dtype: int64


In [6]:
df_combined.head()

,longitude,latitude,Vp_100_kms,MagSus_SI,Density_gcc,SiO2_pct,Al2O3_pct,Fe2O3_pct,MgO_pct,CaO_pct,...,Cobalt,Copper,Diamond,Gold,Graphite,Lithium,Nickel,Rare Earth Elements (REE),Silver,Uranium
0,-133.65358,59.582720,6.336446,0.0,2.847059,54.561337,10.792269,7.341256,3.399487,4.073229,...,0,1,0,1,0,0,0,0,1,0
1,-133.60111,58.735833,6.259152,0.0,2.830975,55.898155,2.291046,6.687087,1.044798,1.865145,...,0,1,0,1,0,0,0,0,1,0
2,-133.52166,59.735833,6.337446,0.0,2.847379,69.490746,12.329194,2.595352,0.593618,1.409079,...,0,1,0,1,0,0,0,0,1,0
3,-133.27033,59.382720,6.339731,0.0,2.848117,57.787693,13.201621,6.252619,2.486531,4.939197,...,0,0,0,1,0,0,0,0,0,0
4,-132.29361,58.210833,6.267930,0.0,2.834029,49.014496,11.270087,25.223366,0.869874,5.072027,...,0,1,0,1,0,0,0,0,1,0


Running feature importance using random forest

In [7]:
import numpy as np

# log transforms
log_cols = ['Au_ppb', 'Cu_ppm', 'Ni_ppm', 'Co_ppm', 'Li_ppm', 
            'Cr_ppm', 'Zn_ppm', 'Pb_ppm', 'Mo_ppm', 'As_ppm',
            'Th_ppm', 'U_ppm', 'La_ppm', 'Ce_ppm', 'Nd_ppm',
            'Nb_ppm', 'V_ppm', 'W_ppm']

for col in log_cols:
    df_combined[f"log_{col}"] = np.log1p(df_combined[col])

# elemental ratios
df_combined["Au_As_ratio"] = df_combined["Au_ppb"] / (df_combined["As_ppm"] + 1e-6)
df_combined["Ni_Co_ratio"] = df_combined["Ni_ppm"] / (df_combined["Co_ppm"] + 1e-6)
df_combined["Cu_Au_ratio"] = df_combined["Cu_ppm"] / (df_combined["Au_ppb"] + 1e-6)
df_combined["La_Ce_ratio"] = df_combined["La_ppm"] / (df_combined["Ce_ppm"] + 1e-6)
df_combined["U_Th_ratio"] = df_combined["U_ppm"] / (df_combined["Th_ppm"] + 1e-6)

# oxide ratios
df_combined["SiO2_Al2O3_ratio"] = df_combined["SiO2_pct"] / (df_combined["Al2O3_pct"] + 1e-6)
df_combined["Fe_Mg_ratio"] = df_combined["Fe2O3_pct"] / (df_combined["MgO_pct"] + 1e-6)
df_combined["alkali_index"] = df_combined["Na2O_pct"] + df_combined["K2O_pct"]

# geophysics interactions
df_combined["density_SiO2"] = df_combined["Density_gcc"] * df_combined["SiO2_pct"]
df_combined["Vp_MgO"] = df_combined["Vp_100_kms"] * df_combined["MgO_pct"]

print("New shape:", df_combined.shape)
print("New columns added:", df_combined.shape[1] - 35)  # 35 = original columns

New shape: (124294, 71)
New columns added: 36


C:\Users\sjade\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\Users\sjade\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\Users\sjade\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\Users\sjade\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pandas\core\arraylike.py:399: RuntimeWarni

In [14]:
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import pandas as pd

# all features including engineered ones
all_feature_cols = feature_cols + [f"log_{col}" for col in log_cols] + [
    "Au_As_ratio", "Ni_Co_ratio", "Cu_Au_ratio", "La_Ce_ratio", "U_Th_ratio",
    "SiO2_Al2O3_ratio", "Fe_Mg_ratio", "alkali_index",
    "density_SiO2", "Vp_MgO"
]

def plot_feature_importance(mineral, top_n=20):
    # get positives and unlabeled
    X_pos = df_combined[df_combined[mineral] == 1][all_feature_cols].values
    X_unl = df_combined[df_combined[mineral] == 0][all_feature_cols].values
    
    # sample unlabeled to balance (5x positives)
    n_sample = min(len(X_pos) * 5, len(X_unl))
    sample_idx = np.random.choice(len(X_unl), n_sample, replace=False)
    X_unl_sample = X_unl[sample_idx]
    
    # combine
    X = np.vstack([X_pos, X_unl_sample])
    y = np.array([1] * len(X_pos) + [0] * len(X_unl_sample))
    
    # train random forest
    rf = RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight="balanced"
    )
    rf.fit(X, y)
    
    # get importances
    importances = pd.Series(rf.feature_importances_, index=all_feature_cols)
    importances = importances.sort_values(ascending=False).head(top_n)
    
   
    return importances

# run for each mineral
top_features_per_mineral = {}

for mineral in target_minerals:
    print(f"\n{mineral}")
    print("-" * 40)
    importances = plot_feature_importance(mineral)
    top_features_per_mineral[mineral] = importances.index.tolist()
    print(importances)


Gold
----------------------------------------
log_La_ppm     0.055719
Density_gcc    0.055656
La_ppm         0.043424
La_Ce_ratio    0.043030
Fe2O3_pct      0.036383
Ce_ppm         0.031450
log_Ce_ppm     0.031189
Nd_ppm         0.027306
W_ppm          0.023208
log_Nd_ppm     0.021154
log_W_ppm      0.021001
K2O_pct        0.019984
TiO2_pct       0.018953
log_Zn_ppm     0.018753
Vp_100_kms     0.017266
log_Th_ppm     0.017239
MnO_pct        0.016754
Na2O_pct       0.015625
log_Cr_ppm     0.014892
U_Th_ratio     0.014833
dtype: float64

Silver
----------------------------------------
Density_gcc    0.073978
Fe2O3_pct      0.047906
log_La_ppm     0.042349
La_ppm         0.041308
Ce_ppm         0.037522
log_Ce_ppm     0.032026
Nd_ppm         0.026319
log_Th_ppm     0.023309
Th_ppm         0.023103
La_Ce_ratio    0.022356
W_ppm          0.021046
Zn_ppm         0.020641
log_W_ppm      0.020309
Fe_Mg_ratio    0.019674
log_Nd_ppm     0.019358
log_Zn_ppm     0.019158
SiO2_pct       0.019048
T

In [8]:
# remove underperforming engineered features
cols_to_drop = ["SiO2_Al2O3_ratio", "alkali_index", "density_SiO2"]
df_combined.drop(columns=cols_to_drop, inplace=True)

# updated feature cols
all_feature_cols = feature_cols + [f"log_{col}" for col in log_cols] + [
    "Au_As_ratio", "Ni_Co_ratio", "Cu_Au_ratio", "La_Ce_ratio", "U_Th_ratio",
    "Fe_Mg_ratio", "Vp_MgO"
]

print("Total features:", len(all_feature_cols))

Total features: 56


Positive Unlabeled Learning (PUL) using Support Vector Classifier and Radial Basis Function Neural Network (RBFNN)

In [9]:
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings("ignore")

# --- updated feature columns ---
base_feature_cols = [
    'Vp_100_kms', 'MagSus_SI', 'Density_gcc',
    'SiO2_pct', 'Al2O3_pct', 'Fe2O3_pct', 'MgO_pct', 'CaO_pct',
    'Na2O_pct', 'K2O_pct', 'TiO2_pct', 'P2O5_pct', 'MnO_pct',
    'Au_ppb', 'Cu_ppm', 'Ni_ppm', 'Co_ppm', 'Li_ppm', 'Cr_ppm',
    'Zn_ppm', 'Pb_ppm', 'Mo_ppm', 'As_ppm', 'Th_ppm', 'U_ppm',
    'La_ppm', 'Ce_ppm', 'Nd_ppm', 'Nb_ppm', 'V_ppm', 'W_ppm'
]

target_minerals = [
    'Gold', 'Silver', 'Copper', 'Nickel',
    'Cobalt', 'Uranium', 'Diamond', 'Lithium'
]

engineered_cols = (
    [f"log_{col}" for col in log_cols] +
    ["Au_As_ratio", "Ni_Co_ratio", "Cu_Au_ratio",
     "La_Ce_ratio", "U_Th_ratio", "Fe_Mg_ratio", "Vp_MgO"]
)

all_feature_cols = base_feature_cols + engineered_cols
print(f"Total features: {len(all_feature_cols)}")



Total features: 56


In [10]:
# check where NaNs are
print("NaNs per column:")
print(df_combined[all_feature_cols].isna().sum()[df_combined[all_feature_cols].isna().sum() > 0])

NaNs per column:
log_Cu_ppm      62
log_Ni_ppm     358
log_Cr_ppm      10
log_Zn_ppm       3
log_Pb_ppm     176
log_Mo_ppm    1503
log_W_ppm       13
dtype: int64


In [11]:
# check for infinities too (can come from division)
import numpy as np

# replace inf and -inf with NaN first
df_combined[all_feature_cols] = df_combined[all_feature_cols].replace([np.inf, -np.inf], np.nan)

# check how many NaNs we have total
print("Total NaNs:", df_combined[all_feature_cols].isna().sum().sum())

# fill NaNs with column median
df_combined[all_feature_cols] = df_combined[all_feature_cols].fillna(
    df_combined[all_feature_cols].median()
)

# verify
print("NaNs after fix:", df_combined[all_feature_cols].isna().sum().sum())

Total NaNs: 2125
NaNs after fix: 0


In [12]:

SPY_RATIO = 0.15
HIDDEN_NEURONS = len(all_feature_cols) * 2
RANDOM_STATE = 42

# --- spy technique ---
def spy_technique(X_pos, X_unl, spy_ratio=0.15, random_state=42):
    np.random.seed(random_state)

    n_spies = max(1, int(len(X_pos) * spy_ratio))
    spy_idx = np.random.choice(len(X_pos), n_spies, replace=False)
    remaining_idx = np.setdiff1d(np.arange(len(X_pos)), spy_idx)

    X_spies = X_pos[spy_idx]
    X_pos_remaining = X_pos[remaining_idx]

    X_mixed = np.vstack([X_unl, X_spies])
    y_mixed = np.array([0] * len(X_unl) + [1] * len(X_spies))

    initial_clf = SVC(kernel='rbf', probability=True, random_state=random_state)
    X_train_init = np.vstack([X_pos_remaining, X_mixed])
    y_train_init = np.array([1] * len(X_pos_remaining) + list(y_mixed))
    initial_clf.fit(X_train_init, y_train_init)

    unl_scores = initial_clf.predict_proba(X_unl)[:, 1]
    spy_scores = initial_clf.predict_proba(X_spies)[:, 1]

    threshold = np.percentile(spy_scores, 15)
    reliable_neg_idx = np.where(unl_scores < threshold)[0]
    X_reliable_neg = X_unl[reliable_neg_idx]

    return X_pos, X_reliable_neg

# --- evaluation ---
def evaluate(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "AUC-ROC": round(roc_auc_score(y_true, y_prob), 4),
        "Precision": round(precision_score(y_true, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_true, y_pred, zero_division=0), 4),
        "F1": round(f1_score(y_true, y_pred, zero_division=0), 4)
    }



In [13]:
# --- main loop ---
all_results = []
all_predictions = {}

for mineral in target_minerals:
    print(f"\n{'='*50}")
    print(f"Training models for: {mineral}")
    print(f"{'='*50}")

    X_pos = df_combined[df_combined[mineral] == 1][all_feature_cols].values
    X_unl = df_combined[df_combined[mineral] == 0][all_feature_cols].values

    # subsample unlabeled set
    MAX_UNLABELED = 10000
    if len(X_unl) > MAX_UNLABELED:
        subsample_idx = np.random.choice(len(X_unl), MAX_UNLABELED, replace=False)
        X_unl = X_unl[subsample_idx]


    print(f"Positives: {len(X_pos)}, Unlabeled: {len(X_unl)}")

    if len(X_pos) < 5:
        print(f"Skipping {mineral} — too few positives")
        continue

    scaler = StandardScaler()
    X_pos_scaled = scaler.fit_transform(X_pos)
    X_unl_scaled = scaler.transform(X_unl)

    X_pos_train, X_pos_test = train_test_split(
        X_pos_scaled, test_size=0.2, random_state=RANDOM_STATE
    )

    X_pos_final, X_reliable_neg = spy_technique(
        X_pos_train, X_unl_scaled, spy_ratio=SPY_RATIO
    )

    print(f"Reliable negatives found: {len(X_reliable_neg)}")

    X_train = np.vstack([X_pos_final, X_reliable_neg])
    y_train = np.array([1] * len(X_pos_final) + [0] * len(X_reliable_neg))

    n_test_neg = len(X_pos_test) * 5
    test_neg_idx = np.random.choice(
        len(X_unl_scaled),
        min(n_test_neg, len(X_unl_scaled)),
        replace=False
    )
    X_test = np.vstack([X_pos_test, X_unl_scaled[test_neg_idx]])
    y_test = np.array([1] * len(X_pos_test) + [0] * len(test_neg_idx))

    mineral_predictions = {}

    # --- SVM ---
    print("Training SVM...")
    svm = SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE)
    svm.fit(X_train, y_train)

    svm_prob = svm.predict_proba(X_test)[:, 1]
    svm_metrics = evaluate(y_test, svm_prob)
    print(f"SVM: {svm_metrics}")

    X_all_scaled = scaler.transform(df_combined[all_feature_cols].values)
    mineral_predictions["SVM"] = svm.predict_proba(X_all_scaled)[:, 1]

    all_results.append({
        "Mineral": mineral,
        "Model": "SVM",
        "Positives": len(X_pos),
        **svm_metrics
    })

    # --- RBFNN ---
    print("Training RBFNN...")
    rbfnn = MLPRegressor(
        hidden_layer_sizes=(HIDDEN_NEURONS,),
        activation='logistic',
        max_iter=500,
        random_state=RANDOM_STATE
    )
    rbfnn.fit(X_train, y_train)

    rbfnn_prob = np.clip(rbfnn.predict(X_test), 0, 1)
    rbfnn_metrics = evaluate(y_test, rbfnn_prob)
    print(f"RBFNN: {rbfnn_metrics}")

    mineral_predictions["RBFNN"] = np.clip(
        rbfnn.predict(X_all_scaled), 0, 1
    )

    all_results.append({
        "Mineral": mineral,
        "Model": "RBFNN",
        "Positives": len(X_pos),
        **rbfnn_metrics
    })

    all_predictions[mineral] = mineral_predictions




Training models for: Gold
Positives: 425, Unlabeled: 10000
Reliable negatives found: 7003
Training SVM...
SVM: {'AUC-ROC': 0.7835, 'Precision': 0.6912, 'Recall': 0.5529, 'F1': 0.6144}
Training RBFNN...
RBFNN: {'AUC-ROC': 0.8377, 'Precision': 0.7231, 'Recall': 0.5529, 'F1': 0.6267}

Training models for: Silver
Positives: 281, Unlabeled: 10000
Reliable negatives found: 8206
Training SVM...
SVM: {'AUC-ROC': 0.8524, 'Precision': 0.8636, 'Recall': 0.3333, 'F1': 0.481}
Training RBFNN...
RBFNN: {'AUC-ROC': 0.9288, 'Precision': 0.9524, 'Recall': 0.3509, 'F1': 0.5128}

Training models for: Copper
Positives: 276, Unlabeled: 10000
Reliable negatives found: 7216
Training SVM...
SVM: {'AUC-ROC': 0.7901, 'Precision': 0.6897, 'Recall': 0.3571, 'F1': 0.4706}
Training RBFNN...
RBFNN: {'AUC-ROC': 0.8494, 'Precision': 0.6667, 'Recall': 0.3929, 'F1': 0.4944}

Training models for: Nickel
Positives: 59, Unlabeled: 10000
Reliable negatives found: 4652
Training SVM...
SVM: {'AUC-ROC': 0.7208, 'Precision': 0.

In [18]:
# --- results ---
df_results = pd.DataFrame(all_results)
print("\n\nFINAL RESULTS SUMMARY")
print("=" * 70)
print(df_results.to_string(index=False))


# --- save predictions ---
df_pred = df_combined[["longitude", "latitude"]].copy()
for mineral in all_predictions:
    for model_name, preds in all_predictions[mineral].items():
        df_pred[f"{mineral}_{model_name}"] = preds

df_pred.to_csv("predictions/pul_predictions.csv", index=False)
print("\nPredictions saved to pul_predictions.csv")



FINAL RESULTS SUMMARY
Mineral Model  Positives  AUC-ROC  Precision  Recall     F1
   Gold   SVM        425   0.7835     0.6912  0.5529 0.6144
   Gold RBFNN        425   0.8377     0.7231  0.5529 0.6267
 Silver   SVM        281   0.8524     0.8636  0.3333 0.4810
 Silver RBFNN        281   0.9288     0.9524  0.3509 0.5128
 Copper   SVM        276   0.7901     0.6897  0.3571 0.4706
 Copper RBFNN        276   0.8494     0.6667  0.3929 0.4944
 Nickel   SVM         59   0.7208     0.0000  0.0000 0.0000
 Nickel RBFNN         59   0.7347     1.0000  0.5000 0.6667
 Cobalt   SVM         44   0.9753     1.0000  0.4444 0.6154
 Cobalt RBFNN         44   0.9185     0.0000  0.0000 0.0000
Uranium   SVM         22   0.7280     0.0000  0.0000 0.0000
Uranium RBFNN         22   0.7800     0.0000  0.0000 0.0000
Diamond   SVM          9   0.9500     0.0000  0.0000 0.0000
Diamond RBFNN          9   0.9500     0.0000  0.0000 0.0000

Predictions saved to pul_predictions.csv


In [20]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import os

# load predictions
df_pred = pd.read_csv("predictions/pul_predictions.csv")
df_mines = pd.read_csv("data_cleaned/mines.csv")

# load canada boundary
canada = gpd.read_file(
    "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
)
canada = canada[canada["NAME"] == "Canada"]
# output directory
os.makedirs("predictions/maps", exist_ok=True)

# minerals and models
target_minerals = [
    'Gold', 'Silver', 'Copper', 'Nickel',
    'Cobalt', 'Uranium', 'Diamond'
]

for mineral in target_minerals:
    print(f"Generating map: {mineral}")

    fig, axes = plt.subplots(1, 2, figsize=(24, 8))
    fig.suptitle(
        f"{mineral} Prospectivity Map",
        fontsize=18,
        fontweight="bold"
    )

    mineral_mines = df_mines[df_mines[mineral] == 1]

    for ax, model_name in zip(axes, ['SVM', 'RBFNN']):
        col = f"{mineral}_{model_name}"

        if col not in df_pred.columns:
            ax.set_title(f"{model_name} — not available")
            continue

        # plot canada boundary
        canada.plot(
            ax=ax,
            color="whitesmoke",
            edgecolor="black",
            linewidth=0.5
        )

        # plot prospectivity scores
        sc = ax.scatter(
            df_pred["longitude"],
            df_pred["latitude"],
            c=df_pred[col],
            cmap="RdYlGn",
            s=0.5,
            alpha=0.7,
            vmin=0,
            vmax=1
        )

        # overlay known mine locations
        ax.scatter(
            mineral_mines["longitude"],
            mineral_mines["latitude"],
            c="black",
            s=15,
            marker="^",
            label=f"Known {mineral} mines",
            zorder=5
        )

        # colorbar
        cbar = plt.colorbar(sc, ax=ax, shrink=0.6, pad=0.02)
        cbar.set_label("Prospectivity Score", fontsize=11)

        # labels
        ax.set_title(model_name, fontsize=14, fontweight="bold")
        ax.set_xlabel("Longitude", fontsize=11)
        ax.set_ylabel("Latitude", fontsize=11)
        ax.legend(loc="lower left", fontsize=9)
        ax.set_xlim(-141, -52)
        ax.set_ylim(41.7, 83)

    plt.tight_layout()
    plt.savefig(
        f"predictions/maps/{mineral}_prospectivity.png",
        dpi=150,
        bbox_inches="tight"
    )
    plt.close()
    print(f"Saved: {mineral}_prospectivity.png")

print("\nAll maps generated.")

Generating map: Gold
Saved: Gold_prospectivity.png
Generating map: Silver
Saved: Silver_prospectivity.png
Generating map: Copper
Saved: Copper_prospectivity.png
Generating map: Nickel
Saved: Nickel_prospectivity.png
Generating map: Cobalt
Saved: Cobalt_prospectivity.png
Generating map: Uranium
Saved: Uranium_prospectivity.png
Generating map: Diamond
Saved: Diamond_prospectivity.png

All maps generated.
